# Sampling Methods Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Uniform and inverse CDF sampling

In [ ]:
```python

import math

import random

def sample_uniform(a, b):

    return a + (b - a) * random.random()

def sample_exponential_inverse_cdf(lam):

    u = random.random()

    return -math.log(u) / lam

In [ ]:
```

Generate 10,000 exponential samples and verify the mean is 1/lambda.

### Step 2: Rejection sampling

In [ ]:
```python

def rejection_sample(target_pdf, proposal_sample, proposal_pdf, M):

    while True:

        x = proposal_sample()

        u = random.random()

        if u < target_pdf(x) / (M * proposal_pdf(x)):

            return x

In [ ]:
```

Use rejection sampling to draw from a truncated normal distribution. Verify the shape by histogramming the samples.

### Step 3: Importance sampling

In [ ]:
```python

def importance_sampling_estimate(f, target_pdf, proposal_pdf, proposal_sample, n):

    total = 0

    for _ in range(n):

        x = proposal_sample()

        w = target_pdf(x) / proposal_pdf(x)

        total += f(x) * w

    return total / n

In [ ]:
```

Estimate E[X^2] under a normal distribution using a uniform proposal. Compare to the known answer (mu^2 + sigma^2).

### Step 4: Monte Carlo estimation of pi

In [ ]:
```python

def monte_carlo_pi(n):

    inside = 0

    for _ in range(n):

        x = random.uniform(-1, 1)

        y = random.uniform(-1, 1)

        if x*x + y*y <= 1:

            inside += 1

    return 4 * inside / n

In [ ]:
```

### Step 5: Metropolis-Hastings MCMC

In [ ]:
```python

def metropolis_hastings(target_log_pdf, proposal_sample, proposal_log_pdf, x0, n_samples, burn_in):

    samples = []

    x = x0

    for i in range(n_samples + burn_in):

        x_new = proposal_sample(x)

        log_alpha = (target_log_pdf(x_new) + proposal_log_pdf(x, x_new)

                     - target_log_pdf(x) - proposal_log_pdf(x_new, x))

        if math.log(random.random()) < log_alpha:

            x = x_new

        if i >= burn_in:

            samples.append(x)

    return samples

In [ ]:
```

Sample from a bimodal distribution (mixture of two Gaussians). Visualize the chain's trajectory.

### Step 6: Gibbs sampling

In [ ]:
```python

def gibbs_sampling_2d(conditional_x_given_y, conditional_y_given_x, x0, y0, n_samples, burn_in):

    x, y = x0, y0

    samples = []

    for i in range(n_samples + burn_in):

        x = conditional_x_given_y(y)

        y = conditional_y_given_x(x)

        if i >= burn_in:

            samples.append((x, y))

    return samples

In [ ]:
```

### Step 7: Temperature sampling

In [ ]:
```python

def softmax(logits):

    max_l = max(logits)

    exps = [math.exp(z - max_l) for z in logits]

    total = sum(exps)

    return [e / total for e in exps]

def temperature_sample(logits, temperature):

    scaled = [z / temperature for z in logits]

    probs = softmax(scaled)

    return sample_from_probs(probs)

In [ ]:
```

Show how temperature changes the output distribution for a set of token logits.

### Step 8: Top-k and top-p sampling

In [ ]:
```python

def top_k_sample(logits, k):

    indexed = sorted(enumerate(logits), key=lambda x: -x[1])

    top = indexed[:k]

    top_logits = [l for _, l in top]

    probs = softmax(top_logits)

    idx = sample_from_probs(probs)

    return top[idx][0]

def top_p_sample(logits, p):

    probs = softmax(logits)

    indexed = sorted(enumerate(probs), key=lambda x: -x[1])

    cumsum = 0

    selected = []

    for token_idx, prob in indexed:

        cumsum += prob

        selected.append((token_idx, prob))

        if cumsum >= p:

            break

    sel_probs = [pr for _, pr in selected]

    total = sum(sel_probs)

    sel_probs = [pr / total for pr in sel_probs]

    idx = sample_from_probs(sel_probs)

    return selected[idx][0]

In [ ]:
```

### Step 9: Reparameterization trick

In [ ]:
```python

def reparam_sample(mu, sigma):

    epsilon = random.gauss(0, 1)

    return mu + sigma * epsilon

def reparam_gradient(mu, sigma, epsilon):

    dz_dmu = 1.0

    dz_dsigma = epsilon

    return dz_dmu, dz_dsigma

In [ ]:
```

Demonstrate that gradients flow through the reparameterized sample but not through direct sampling.

### Step 10: Gumbel-Softmax

In [ ]:
```python

def gumbel_sample():

    u = random.random()

    return -math.log(-math.log(u))

def gumbel_softmax(logits, temperature):

    gumbels = [math.log(p) + gumbel_sample() for p in logits]

    return softmax([g / temperature for g in gumbels])

In [ ]:
```

Show how decreasing temperature makes the output approach a one-hot vector.

Full implementations with all visualizations are in `code/sampling.py`.

## Exercises

In [ ]:
1. Implement inverse CDF sampling for the Cauchy distribution. The CDF is F(x) = 0.5 + arctan(x)/pi. Generate 10,000 samples and plot the histogram against the true PDF. Notice the heavy tails (extreme values far from center).

2. Use rejection sampling to generate samples from a Beta(2, 5) distribution using a Uniform(0, 1) proposal. Plot the accepted samples against the true Beta PDF. What is the theoretical acceptance rate?

3. Estimate the integral of sin(x) from 0 to pi using Monte Carlo with 1,000, 10,000, and 100,000 samples. Compare the error at each level. Verify that the error scales as O(1/sqrt(N)).

4. Implement Metropolis-Hastings to sample from a 2D distribution p(x, y) proportional to exp(-(x^2 * y^2 + x^2 + y^2 - 8*x - 8*y) / 2). Plot the samples and the chain trajectory. Experiment with different proposal standard deviations.

5. Build a complete text generation demo: given a vocabulary of 10 words with logits, generate sequences of 20 tokens using (a) greedy, (b) temperature=0.7, (c) top-k=3, (d) top-p=0.9. Compare the diversity of outputs across 5 runs.